In [8]:
import torch
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
import torch
from torchvision.models import ResNet18_Weights
from PIL import Image
import pandas as pd

# Define the transform (same as training)
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def create_model(num_classes):
    """
    Create an EfficientNet-B0 model with dropout and a customized final layer.

    Args:
        num_classes (int): Number of output classes.

    Returns:
        torch.nn.Module: Customized EfficientNet-B0 model.
    """
    # Load EfficientNet-B0 with pretrained weights
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last two feature blocks (for more learning capacity)
    for param in model.features[-4:].parameters():
        param.requires_grad = True

    # Customize the final fully connected layer with dropout
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(model.classifier[1].in_features, num_classes)
    )

    return model
# Initialize the model


# Create sample input for tracing (batch_size=1, channels=3, height=224, width=224)
dummy_input = torch.randn(1, 3, 224, 224)

# Load and prepare model
model = create_model(num_classes=18)
model.load_state_dict(torch.load('best_model_7.pth', map_location=torch.device('cpu')))
model.eval()  # Crucial for dropout/batchnorm layers

# Trace the model with example input
traced_model = torch.jit.trace(model, dummy_input)

# Optimize for mobile (important!)
optimized_model = torch.jit.optimize_for_inference(traced_model)

# Save with .pt extension
optimized_model.save("model.pt")

C:\Users\rkhaz\AppData\Local\Temp\ipykernel_28056\2477034430.py:58: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_7.pth', map_lo

In [ ]:
import torch
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

# Load your PyTorch model
model = create_model(num_classes=18)  # Use your existing create_model function
model.load_state_dict(torch.load('best_model_7.pth', map_location='cpu'))
model.eval()

# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = "model.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=13,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Convert ONNX to TensorFlow
onnx_model = onnx.load(onnx_path)
tf_rep = prepare(onnx_model)
tf_rep.export_graph("tf_model")

# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_saved_model("tf_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

C:\Users\rkhaz\AppData\Local\Temp\ipykernel_28056\3451188018.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_7.pth', map_loc

INFO:tensorflow:Assets written to: tf_model\assets


INFO:tensorflow:Assets written to: tf_model\assets


In [9]:
import torch

# Assume you have a model defined and loaded with weights.

model = create_model(num_classes=18)
model.load_state_dict(torch.load('best_model_7.pth', map_location='cpu'))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224)  # Example input; must match your model's expected input shape.
traced_script_module = torch.jit.trace(model, dummy_input)

# Save the TorchScript model.
traced_script_module.save("model_1.pt")




C:\Users\rkhaz\AppData\Local\Temp\ipykernel_28056\3820699440.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_7.pth', map_loc

^C
